# 20 · Mapping Review UI Walkthrough (v0.3.1)

Walk through Portiere's Streamlit-based **Mapping Review UI** end-to-end:

1. Materialize a small project with a schema mapping.
2. Apply review decisions programmatically (approve / override / reject) using the same `state.py` helpers the Streamlit pages call into.
3. Inspect the persisted `schema_mapping_reviewed.json`.

**Why not start Streamlit from a notebook?** Streamlit runs its own event loop and can't be embedded in a notebook cleanly. The notebook exercises the **state helpers** (the testable surface). To see the UI itself, run `portiere review <project-dir>` in a separate terminal.

Install: `pip install "portiere-health[review]"`.

In [ ]:
import json, shutil
from pathlib import Path

import yaml

import portiere
from portiere.review_ui.state import (
    apply_user_decision,
    load_schema_mapping,
    save_reviewed_schema_mapping,
)

print('portiere version:', portiere.__version__)

## 1 · Materialize a tiny project

Real Portiere projects write `schema_mappings/schema_mapping.yaml` after `project.map_schema(source)`. We'll build that layout by hand for the demo.

In [ ]:
project_dir = Path('/tmp/portiere-review-nb20')
if project_dir.exists():
    shutil.rmtree(project_dir)
(project_dir / 'schema_mappings').mkdir(parents=True)

items = [
    {
        'source_table': 'patients',
        'source_column': 'patient_id',
        'target_table': 'person',
        'target_column': 'person_id',
        'confidence': 0.99,
        'status': 'auto_accepted',
        'candidates': [],
    },
    {
        'source_table': 'patients',
        'source_column': 'dob',
        'target_table': 'person',
        'target_column': 'birth_datetime',
        'confidence': 0.72,
        'status': 'needs_review',
        'candidates': [
            {'target_table': 'person', 'target_column': 'year_of_birth', 'score': 0.65},
        ],
    },
    {
        'source_table': 'encounters',
        'source_column': 'visit_type',
        'target_table': 'visit_occurrence',
        'target_column': 'visit_concept_id',
        'confidence': 0.55,
        'status': 'needs_review',
        'candidates': [],
    },
]
(project_dir / 'schema_mappings' / 'schema_mapping.yaml').write_text(yaml.dump(items))
print(f'Project at: {project_dir}')
print(f'  Items: {len(items)}')

## 2 · Load via the same code path the UI uses

`load_schema_mapping` prefers `schema_mapping_reviewed.json` when it exists; otherwise it falls back to the original YAML. On a fresh project, we get the YAML.

In [ ]:
mapping = load_schema_mapping(project_dir)
print(f'Loaded {len(mapping.items)} items:')
for i, item in enumerate(mapping.items):
    print(
        f'  [{i}] [{item.status.value:<14}] '
        f'{item.source_table}.{item.source_column} -> '
        f'{item.target_table}.{item.target_column}  '
        f'(conf={item.confidence:.2f})'
    )

## 3 · Apply reviewer decisions

These three calls are exactly what the Streamlit page does when a reviewer clicks Approve / Override / Reject.

In [ ]:
# Approve the auto-accepted item (rubber-stamping)
mapping = apply_user_decision(mapping, index=0, decision='approve')

# Override the second item — the AI picked birth_datetime but we want year_of_birth
mapping = apply_user_decision(
    mapping,
    index=1,
    decision='override',
    target_table='person',
    target_column='year_of_birth',
)

# Reject the third item — visit_type doesn't cleanly map
mapping = apply_user_decision(mapping, index=2, decision='reject')

for i, item in enumerate(mapping.items):
    eff_target = f'{item.effective_target_table}.{item.effective_target_column}'
    print(f'  [{i}] {item.status.value:<11} -> {eff_target}')

## 4 · Persist to `schema_mapping_reviewed.json`

The original YAML is never modified.

In [ ]:
out_path = save_reviewed_schema_mapping(mapping, project_dir)
print(f'Wrote: {out_path}')
print()
print('Original YAML still intact:')
print(
    (project_dir / 'schema_mappings' / 'schema_mapping.yaml').read_text()
)

In [ ]:
print('Reviewed JSON contents:')
data = json.loads(out_path.read_text())
print(json.dumps(data, indent=2)[:600] + '\n  ...')

## 5 · Re-loading sees the reviewed decisions

Downstream ETL code that calls `load_schema_mapping` now gets the reviewed targets, not the AI's originals.

In [ ]:
mapping2 = load_schema_mapping(project_dir)
for i, item in enumerate(mapping2.items):
    print(
        f'  [{i}] {item.status.value:<11} '
        f'{item.effective_target_table}.{item.effective_target_column}'
    )

## 6 · Launch the actual UI

In a separate terminal:

```bash
portiere review /tmp/portiere-review-nb20
```

You'll see the same items in a browser at http://127.0.0.1:8501, with the prior decisions already loaded from the reviewed JSON we just wrote.

Concept-mapping review is coming in v0.3.2; for now the sidebar shows only the Schema Mapping page.

Full reference: [`docs/mapping-review-ui.md`](../mapping-review-ui.md).

In [ ]:
# Tidy up
shutil.rmtree(project_dir)
print('Cleaned up.')